# Synthetic: many small inputs, many derived statements

A shape seen in user testing: a folder of thousands of small text files read into one
frame, then statement after statement derived from it. Every derived statement inherits
every file, so per-file bookkeeping multiplies by statements (before d8c9f31 each save,
lookup and upstream check re-hashed all of them: 19-108 s per cell against a 25 s
uncached notebook).

```
python benchmarks/bench_notebook_overhead.py benchmarks/synthetic_many_inputs.ipynb --mode off
python benchmarks/bench_notebook_overhead.py benchmarks/synthetic_many_inputs.ipynb --mode cold
```

In [ ]:
# Inputs are generated once per machine, into the temp dir, and aged a day:
# real exports are old files, and cash treats a file written moments ago
# differently from one untouched since this morning.
import os, tempfile, time
from pathlib import Path

DOCS = Path(tempfile.gettempdir()) / "cash_bench_many_inputs"
N_FILES = 3000
if not DOCS.exists() or len(os.listdir(DOCS)) != N_FILES:
    DOCS.mkdir(exist_ok=True)
    day_ago = time.time() - 86_400
    for i in range(N_FILES):
        p = DOCS / "doc{:05d}.txt".format(i)
        p.write_text(f"Title {i % 97}\n" + " ".join(f"word{(i * j) % 211}" for j in range(20 + i % 80)))
        os.utime(p, (day_ago, day_ago))
print(N_FILES, "inputs in", DOCS)

In [ ]:
import pandas as pd

def read_doc(path):
    raw = path.read_text(encoding='utf-8')
    title, _, body = raw.partition('\n')
    return {'file': path.name, 'title': title, 'body': body, 'bytes': len(raw)}

paths = sorted(DOCS.iterdir())
docs = pd.DataFrame([read_doc(p) for p in paths])
print(len(docs), 'docs')

In [ ]:
docs['text'] = docs['body'].str.lower()
docs['n_words'] = docs['text'].str.split().str.len()
docs = docs[docs['n_words'] >= 5].reset_index(drop=True)
by_title = docs.groupby('title')['n_words'].sum()
long_docs = docs[docs['n_words'] > docs['n_words'].median()]
print(len(docs), len(by_title), len(long_docs))

In [ ]:
vocab = docs['text'].str.split().explode().value_counts()
top = vocab.head(50)
per_file = docs.set_index('file')['n_words']
sizes = docs['bytes'].describe()
summary = {'docs': len(docs), 'words': int(per_file.sum()), 'top': top.index[0]}
print(summary)